# カスタムLoRA: LLaMA3-8B-Instructのファインチューニング
---

**概要:** 

このNotebookでは、Low-Rank Adaptation (LoRA)を使ってLLaMA 3モデルをファインチューニングするプロセスを探ります。扱うトピックは以下の通りです：

1. **データ前処理**: モデルのトレーニングに最適な入力を確保するためのデータの準備とクリーニング
2. **学習のセットアップ**:
   - **Tokenizerのロード**: テキストデータを処理するために、LLaMA 3に適したトークナイザーを初期化
   - **ハイパーパラメータの設定**: 効果的なモデルトレーニングのための主要なハイパーパラメータを定義
   - **PEFT (Parameter-Efficient Fine-Tuning)**:  少ない計算ソースでモデルをファインチューニングする技術
   - **量子化**: パフォーマンスを維持したままモデルサイズを縮小し、より高速な推論とメモリ使用量の削減が可能
   - **事前学習済みモデルのロード**: 事前学習されたLLaMA 3モデルを持ち込み、カスタムデータでさらにファインチューニング
   - **Trainerのハイパーパラメータの設定**: 最適なパフォーマンスを得るために、トレーニングループに特有のパラメータを設定
   - **推論の実行**: ファインチューニングされたモデルを新しいデータで評価し、その性能と精度をテストを実施

このNotebookは、LlaMA 3モデルを特定のユースケースに合わせて効果的にファインチューニングし、最適化するための手順を説明します。

## データ前処理
トレーニングにはopenassistantのguanacoデータセットを使用します。

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from datasets import load_dataset

In [ ]:
ds = load_dataset("timdettmers/openassistant-guanaco")

データセットから5つのサンプルを見てみましょう。

In [ ]:
for sample in ds['train'].select(range(5)):
    print(f"\n {'*' * 64}\n{sample}\n{'*' * 64}")

`text`キーを持つdictオブジェクトを複数の言語で見ることができます。このラボでは、英語のサンプルに限定します。

In [ ]:
from langdetect import detect

def remove_nonEnglish_rows(ds):
    # Initialize an empty list to store rows detected as English
    new_ds = []
    
    # Initialize a list to store indices of rows that cause issues (corner cases)
    corner_case = []
    
    # Iterate through each row in the dataset's 'text' column
    for i, row in enumerate(ds['text']):
        try:
            # Detect the language of the text
            if detect(str(row)) == 'en':
                # If the language is English, add the row to new_ds
                new_ds.append(row)
        except:
            # If an exception occurs, add the index to corner_case
            corner_case.append(i)
    
    # Return the list of English rows and the indices of corner cases
    return new_ds, corner_case


In [ ]:
filter_train_samples,cc_train = remove_nonEnglish_rows(ds['train'])

print("Count of training samples: ",len(filter_train_samples))

In [ ]:
filter_test_samples,cc_test = remove_nonEnglish_rows(ds['test'])
print("Count of testing samples: ",len(filter_test_samples))


In [ ]:
# save English text samples
import json
def save_jsonl(ds,filename):
    with open(f"data/{filename}.jsonl", "w") as write_file:
            json.dump(ds, write_file, indent=4)
            print("dataset saved in jsonl format ....")

LLaMA3モデルは，学習データセットが[こちら](https://llama.meta.com/docs/model-cards-and-prompt-formats/meta-llama-3/)のような特定の形式であることを期待します。

In [ ]:
def transform_to_template(example):
    conversation_text = example['text']
    segments = conversation_text.split("###")[1:]
    

    for idx,segment in enumerate(segments):
        if idx%2==0:
            segments[idx] = segment.replace('Human:',"<|start_header_id|>user<|end_header_id|>") + "<|eot_id|>"
        else:
            segments[idx] = segment.replace('Assistant:',"<|start_header_id|>assistant<|end_header_id|>") + "<|eot_id|>"
    
    

    segments = ["<|begin_of_text|><|start_header_id|>system<|end_header_id|> You are a helpful assistant<|eot_id|>"] + segments

    return {'text': ''.join(segments)}

これらのデータセットを保存するための別のフォルダを作ります。

In [ ]:
! mkdir -p data
! mkdir -p data/filtered

In [ ]:
# set file names  
save_train_filename = 'filtered/train'
save_test_filename = 'filtered/test'

# save file
save_jsonl(filter_train_samples, save_train_filename)
save_jsonl(filter_test_samples, save_test_filename)

In [ ]:
dataset = load_dataset('data/filtered/')

In [ ]:
template_dataset = dataset.map(transform_to_template)

In [ ]:
!mkdir -p data/ds_preprocess
template_dataset.save_to_disk('data/ds_preprocess/')

これで変換されたデータセットを保存できました。これで、カーネルが故障した場合にデータセットを直接ロードするのに役立ちます。

## 学習セットアップ

ファインチューニングの設定には、以下のライブラリーが役に立ちます。

In [ ]:
# In some cases where you have access to limited computing resources, you might have to uncomment os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:64" if you run into not enough memory issue 

import os
import torch
import json
import re

from peft import LoraConfig, PeftModel
from trl import SFTTrainer
from datasets import load_dataset, load_from_disk
from langdetect import detect
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
)

# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:64"

重要な成果物のロードと保存のための重要なパスを設定します。

Llama-3ファミリーのモデルはオープンソースですが、アクセスリクエストの承認が必要です。ブートキャンプ環境では、ウエイトはすでにhuggingface互換フォーマットに変換され、参加者が素早くアクセスできるように共有の場所に保存されています。

あなた自身の環境でこの教材を実行する場合は、[こちら](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct)からLlamaモデルへのアクセスをリクエストし、こちらの[リンク](https://huggingface.co/settings/tokens)からHuggingFaceユーザアクセストークンを生成してください。

In [ ]:
# initialize path to the base model 
# base_model = "meta-llama/Meta-Llama-3-8B-Instruct" # Use this while running the material in your own standalone environment.

base_model = "/mnt/lustre/docker-strg/llama3_8b_weights" # shared model weight location

# set the path to the dataset template
data_path = "data/ds_preprocess/train"
# set the path to the dataset template
eval_path = "data/ds_preprocess/test"

# load the transformed dataset
dataset = load_from_disk(data_path)
eval_dataset = load_from_disk(eval_path)

In [ ]:
# Needed for standalone run
# token='hf_..'

### Tokenizerのロード

In [ ]:
#The below code should be used to download the tokenizer for the Llama-3-8V-Instruct model, once you get the permissions.
tokenizer = AutoTokenizer.from_pretrained(base_model,
                                          # token=token,
                                          # trust_remote_code=True
                                         )

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

結果を保存するディレクトリも必要です。

In [ ]:
! mkdir -p model

### 学習のハイパーパラメータの設定

In [ ]:
training_params = TrainingArguments(
    output_dir="model/results",             # Directory to save the model results
    num_train_epochs=2,                     # Number of training epochs
    per_device_train_batch_size=5,          # Batch size per device during training
    gradient_accumulation_steps=4,          # Number of steps to accumulate gradients
    group_by_length=True,                   # Group sequences of similar lengths together
    save_steps=100,                         # Save model checkpoint every 100 steps
    logging_steps=25,                       # Log metrics every 25 steps
    learning_rate=2e-4,                     # Initial learning rate
    weight_decay=0.001,                     # Weight decay to apply (L2 regularization)
    fp16=False,                             # Use 16-bit precision (half-precision) during training
    bf16=False,                             # Use bfloat16 precision
    max_grad_norm=0.3,                      # Maximum gradient norm (for gradient clipping)
    max_steps=-1,                           # Total number of training steps (-1 means no limit)
    warmup_ratio=0.03,                      # Ratio of steps to perform learning rate warmup
    optim="paged_adamw_32bit",              # Optimizer to use (32-bit AdamW with paged memory)
    lr_scheduler_type="constant",           # Learning rate scheduler type (constant)
    report_to="tensorboard"                 # Reporting tool (TensorBoard in this case)
)


### PEFT
LoRA テクニックは `LoraConfig` によって適用され、ベースモデルへの適用方法を制御する PEFT パラメータを提供します。以下のセルで使用されているパラメータの説明は以下の通りです：

- **lora_alpha**： LoRA スケーリング係数
- **lora_dropout**： LoRAレイヤーのドロップアウト確率。
- **r**: 更新行列のランク、int で表される。ランクが低いほど更新行列が小さくなり、学習可能なパラメータが少なくなります。
- **bias**： バイアスパラメータを学習するかどうかを指定する。'none'、'all'、'lora_only'のいずれかを指定します。
- **task_type**： `CAUSAL_LM`, `FEATURE_EXTRACTION`, `QUESTION_ANS`, `SEQ_2_SEQ_LM`, `SEQ_CLS and TOKEN_CLS` などがあります。  

実行したいタスクはテキスト生成なので、task_type をテキスト生成タスクでよく使われる Causal language model `(CAUSAL_LM)` に設定しました。以下のセルを実行して、LoRAの設定を行ってください。

In [ ]:
peft_params = LoraConfig(
    lora_alpha=16,                # Alpha parameter for Lora scaling
    lora_dropout=0.1,             # Dropout rate for Lora layers
    r=64,                         # Rank of the Lora matrices
    bias="none",                  # Type of bias to apply (none in this case)
    task_type="CAUSAL_LM",        # Type of task (Causal Language Modeling in this case)
)


### 量子化
**4-bit 量子化コンフィグレーション**

モデルの量子化は、一般的なディープラーニングの最適化手法であり、モデルデータ（ネットワークパラメータと活性化）を浮動小数点から低精度表現（通常は8ビット整数）に変換します。量子化はより少ないビット数でデータを表現するため、特に大規模な言語モデル（LLM）において、メモリ使用量を削減し、推論を高速化するのに有効な手法です。PEFTメソッドと組み合わせることで、推論のためのLLMの学習とロードを容易にすることができます。

<center><img src="imgs/quantization.png" height="400px" width="700px" /></center>
<center> <a href="https://developer.nvidia.com/blog/achieving-fp32-accuracy-for-int8-inference-using-quantization-aware-training-with-tensorrt/" > source: Using Quantization Aware Training with NVIDIA TensorRT</a></center>

モデルを量子化するいくつかの方法とアルゴリズムは[こちら](https://huggingface.co/docs/peft/main/en/developer_guides/quantization)にあります。量子化を簡単に実装し、Transformersと統合するためのライブラリに `bitsandbytes` ライブラリがあります。このライブラリは、`BitsAndBytesConfig`クラスを使ってモデルを8ビットまたは4ビットに量子化するための設定パラメータを提供します。以下のセルで使用されている4ビットのパラメータは以下のように記述されています：

- **load_in_4bit**: モデルをロードする際に4ビットに量子化するには `True` を設定。
- **bnb_4bit_quant_type**: `"nf4"` に設定すると、正規分布から初期化された重みに特別な4ビットのデータ型を使用。
- **bnb_4bit_use_double_quant**: `True` に設定すると、既に量子化された重みをネストした量子化スキームで量子化。
- **bnb_4bit_compute_dtype**: `torch.float16` または `torch.bfloat16` に設定。

以下のセルを実行して、モデルの4ビット量子化を設定。


In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)

### 事前学習モデルのロード

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=quant_config,
    device_map={"": 0},
    # token=token
)
model.config.use_cache = False
model.config.pretraining_tp = 1

In [ ]:
!nvidia-smi

モデルがGPUにロードされていることが分かります。

### Trainerのハイパーパラメータのセット

モデルトレーナを開始するために、 [Supervised fine-tuning (SFT)](https://huggingface.co/docs/trl/en/sft_trainer)からトレーナオブジェクトを作成します。SFTは強化学習を使って変換言語モデルを学習するための統合変換ツール [Reinforcement Learning (TRL)](https://huggingface.co/docs/trl/en/index)の一部です。その他に、[Reward Modeling step (RM)](https://huggingface.co/docs/trl/en/reward_trainer) や [Proximal Policy Optimization (PPO)](https://arxiv.org/abs/1707.06347)があります。SFTトレーナーオブジェクトでは、モデル、学習データセット、PEFT設定オブジェクト、モデル・トークナイザー、学習引数パラメータを設定します。また、データセット内で使用するフィールド（`text`）も指定します。
**注意:** *単一のDGX A100 GPUで実行する場合は、`max_seq_length`の値を1024に変更するか、デフォルトのnoneに設定します。*

In [ ]:
len(dataset)

In [ ]:
# The notebook uses a 1000 random samples for training in the interest of time.
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset.shuffle(seed=42).select(range(1000)),
    eval_dataset = eval_dataset.select(range(len(eval_dataset))),
    dataset_text_field="text",
    peft_config=peft_params,
    args=training_params,
    max_seq_length=1024,
    packing=False,
)

以下のセルを実行して、以下のSFT trainerオブジェクトの学習を行います。

In [ ]:
import jinja2
print(jinja2.__version__)


In [ ]:
trainer.train()

In [ ]:
# save model
new_model = "model/Llama-3-8b-instruct-hf-finetune"
trainer.model.save_pretrained(new_model)
trainer.tokenizer.save_pretrained(new_model)

In [ ]:
trainer.model.save_pretrained(new_model,safe_serialization=False)

### 推論の実行

In [ ]:
from transformers import pipeline

In [ ]:
pp= pipeline(model=model, tokenizer=tokenizer, max_length=200, task="text-generation")

In [ ]:
print(pp("what has research identified in potential monopsonies"))

In [ ]:
print(pp("what are some artists similar to Dvorak?"))

In [ ]:
model = PeftModel.from_pretrained(model, new_model)
model = model.merge_and_unload()

In [ ]:
# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
tokenizer.add_special_tokens({'pad_token': '[PAD]'})
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
pp= pipeline(model=model, tokenizer=tokenizer, max_length=200, task="text-generation")

In [ ]:
!nvidia-smi

以下のコマンドは、現在のカーネルを終了させ、NIMを実行するためにGPUを解放します。

In [ ]:
!kill -9 $(nvidia-smi --query-compute-apps=pid --format=csv,noheader | awk 'NR==1')


カスタムLoRAアダプターを使用するには、<a href="nim_lora_adapter.ipynb"> nim_lora_adapter</a> notebookにアクセス下さい。

---
## Licensing

Copyright © 2024 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.